# Lab 11 — Flash attention and KV cache

*MSGAI · Lecture 11 · 90 minutes*

Lecture 11 is about making attention cheap at inference time. Do not store what you do not need (Flash attention).
Do not recompute what you already have (KV cache). Keep what you store small (GQA, MLA), and share it between
conversations (prefix caching). In this lab you build each of these in about 15 lines and measure what it buys.

The model is the tiny character-level GPT of Lab 10 (4 layers, 4 heads, 128 dimensions, trained on Shakespeare with a
context of **L = 64 characters**, RoPE positions). Its code is in this notebook (Exercise 2) and its trained weights
are in `checkpoints/rope.pt`. If your machine has a GPU, the whole notebook runs on it.

**How to work**

1. A cell that contains `# YOUR CODE HERE` is yours: replace the `raise NotImplementedError()` by 5 to 20 lines of code.
2. Run the cell below it. It checks your code with a few `assert`s: if it runs without an error, your code is right.
3. Answer each question in the *Your answer here* cell. Two or three sentences are enough.

Stuck for more than five minutes? Ask for help.

## Setup

In [ ]:
import copy
import math
import statistics
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)                         # every tensor of this notebook is created on this device
torch.set_grad_enabled(False)                            # this lab runs models, it never trains them
torch.set_num_threads(min(4, torch.get_num_threads()))   # on a CPU, tiny tensors: more threads only add overhead

# TinyShakespeare, one token per character
text = open("data/input.txt").read()
chars = sorted(set(text))                                # the vocabulary: the 65 characters of the text
stoi = {c: i for i, c in enumerate(chars)}               # character -> id


def encode(s):
    return torch.tensor([stoi[c] for c in s])


def decode(ids):
    return "".join(chars[i] for i in ids.tolist())


data = encode(text)
val_data = data[int(0.9 * len(data)) :]                  # the last 10% of the text: the models were not trained on it
L = 64                                                   # L_train: the context length the models were trained with
print(f"device: {device}")

The two RoPE functions of Lab 10:

In [ ]:
def rope_theta(head_dim, base=10000.0):
    """The frequencies of RoPE, shape (head_dim/2,): theta_i = base^(-2i/head_dim) for the pair (2i, 2i+1)."""
    return base ** (-torch.arange(0, head_dim, 2) / head_dim)


def apply_rope(x, angles):
    """Rotate each pair (x[..., 2i], x[..., 2i+1]) of token t by angles[t, i].   x (..., T, head_dim), angles (T, head_dim/2)"""
    x_even, x_odd = x[..., 0::2], x[..., 1::2]
    cos, sin = angles.cos(), angles.sin()
    out = torch.empty_like(x)
    out[..., 0::2] = x_even * cos - x_odd * sin
    out[..., 1::2] = x_even * sin + x_odd * cos
    return out

## Exercise 1 — Online softmax (Flash attention)

> Lecture 11, *Flash attention*.

A GPU has a lot of slow memory and very little fast memory. On an A100, the main memory (HBM) holds 40 to 80 GB. The
on-chip memory (SRAM) holds about 20 MB in total and is roughly ten times faster. Standard attention writes the whole
matrix of scores to HBM, reads it back for the softmax, writes the weights, and reads them again to multiply by $V$.
FlashAttention works on chunks of keys and values that fit in SRAM and never stores that matrix.

The obstacle is the softmax: its denominator is a sum over *all* keys. The **online softmax** removes the obstacle.
Take one query $q$ and $L$ keys, read in chunks, with scores $a_i=q\cdot k_i/\sqrt{d}$. Keep three running quantities:
$d_k$, the largest score so far; $s_k$, the sum of $e^{a_i-d_k}$ so far; $o_k$, the output so far.
Start from $d_0=-\infty$, $s_0=0$, $o_0=0$, and for chunk $k$:

$$\begin{aligned}
d_k&=\max\big(d_{k-1},\ \max_{i\in\text{chunk }k}a_i\big)\\
s_k&=s_{k-1}\,e^{\,d_{k-1}-d_k}+\sum_{i\in\text{chunk }k}e^{\,a_i-d_k}\\
o_k&=o_{k-1}\,\frac{s_{k-1}}{s_k}\,e^{\,d_{k-1}-d_k}+\sum_{i\in\text{chunk }k}\frac{e^{\,a_i-d_k}}{s_k}\,v_i
\end{aligned}$$

After the last chunk, $o_B=\operatorname{softmax}(a)\,V$.

### 1a. Why subtract the maximum?

In [ ]:
a = torch.tensor([1.0, 50.0, 100.0])            # three scores, in float32
print("exp(a)                  =", torch.exp(a).tolist())      # the largest float32 is 3.4e38 = exp(88.7)
print("softmax, naive          =", (torch.exp(a) / torch.exp(a).sum()).tolist())
shifted = torch.exp(a - a.max())                # the largest term is exp(0) = 1
print("softmax, max subtracted =", (shifted / shifted.sum()).tolist())

### 1b. The baseline: all weights at once

In [ ]:
def attention_naive(q, K, V):
    """Attention of one query over L keys, the standard way.

    q (d,)    K (L, d)    V (L, d_v)    returns (d_v,):  sum_i softmax(a)_i v_i   with a_i = q . k_i / sqrt(d)
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
q = torch.tensor([2.0, 0.0, 0.0, 0.0])                    # d = 4, so sqrt(d) = 2
K = torch.tensor([[1.0, 0.0, 0.0, 0.0],
                  [0.0, 0.0, 0.0, 0.0],
                  [-1.0, 0.0, 0.0, 0.0]])                 # the scores are a = (1, 0, -1)
V = torch.eye(3)                                          # with V = identity, the output is softmax(a) itself
assert torch.allclose(attention_naive(q, K, V), torch.tensor([0.6652, 0.2447, 0.0900]), atol=1e-4)

### 1c. The online softmax

Read `K` and `V` in slices of `chunk` rows. Between two chunks you may only keep $d$, $s$ and $o$.
(The last chunk is shorter when `chunk` does not divide $L$.)

In [ ]:
def attention_online(q, K, V, chunk):
    """The same result as attention_naive, reading `chunk` keys and values at a time.

    Lecture notation: d = running maximum of the scores, s = running sum of exp(a_i - d), o = running output.
    Never build a tensor that has one entry per key.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
q, K, V = torch.randn(16), torch.randn(100, 16), torch.randn(100, 8)
for chunk in [1, 7, 32, 100, 1000]:                       # 7 and 32 do not divide 100: the last chunk is shorter
    assert torch.allclose(attention_online(q, K, V, chunk), attention_naive(q, K, V), atol=1e-5)
assert torch.allclose(attention_online(q, 100 * K, V, 7), attention_naive(q, 100 * K, V), atol=1e-4)     # scores of a few hundred
assert torch.allclose(attention_online(torch.ones(16), K - 80, V, 7), attention_naive(torch.ones(16), K - 80, V),
                      atol=1e-4)                          # all the scores around -320

Is it faster? Try 200,000 keys:

In [ ]:
torch.manual_seed(0)
dim, n_keys = 64, 200_000
q, K, V = torch.randn(dim), torch.randn(n_keys, dim), torch.randn(n_keys, dim)

for name, fn in (("naive", lambda: attention_naive(q, K, V)),
                 ("online, chunks of 1024", lambda: attention_online(q, K, V, 1024))):
    runs = []
    for _ in range(3):                                    # best of 3: the first run also warms up the device
        t0 = time.perf_counter()
        out = fn()[:3].tolist()                           # .tolist() waits until the result is there, also on a GPU
        runs.append(time.perf_counter() - t0)
    print(f"{name:24s}{1e3 * min(runs):8.1f} ms     output[:3] = {[round(x, 5) for x in out]}")

### 1d. Optional: all the queries at once

Real attention has many queries. Each needs its own $d$, $s$ and $o$, but one pass over the keys can serve them all.
Skip this if you are short of time.

In [ ]:
def attention_online_batched(Q, K, V, chunk):
    """Online softmax for n queries at once.   Q (n, d)   K (L, d)   V (L, d_v)   returns (n, d_v)."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
Q, K, V = torch.randn(6, 16), 20 * torch.randn(77, 16), torch.randn(77, 8)
one_by_one = torch.stack([attention_naive(q, K, V) for q in Q])
for chunk in [1, 10, 77, 500]:
    assert torch.allclose(attention_online_batched(Q, K, V, chunk), one_by_one, atol=1e-4)

**Question 1.**
1. Our online version is slower than the naive one. Why? What does a real FlashAttention kernel save on a GPU?
2. While one chunk is processed, which quantities are in SRAM? Which quantity of standard attention never exists at all?

*Your answer here*

## Exercise 2 — KV cache

> Lecture 11, *KV cache*: prefill and decode.

Without a cache, generating token $t+1$ means running the model on all $t$ tokens again. But attention is causal: a token
never looks at later tokens, so the keys and values of the first $t-1$ tokens are exactly what they were one step
earlier. The KV cache keeps them.

- **Prefill.** Process all the tokens of the prompt in parallel and keep the keys and values of every layer.
- **Decode.** Process only the newest token: its query attends over the cached keys and its own.

### 2a. The attention step with a cache

Every layer of the model projects the **new** tokens to `q`, `k`, `v` and hands them to an *attention step*, together
with the cache of that layer. You write that step: it is the attention of the model.

**Your task.** Write the attention step of the RoPE model. For the rotation use the two functions of Lab 10 from the
setup: the angle of the pair $i$ at position $m$ is $m\,\theta_i$, with $\theta$ = `rope_theta(head_dim)`.

In [ ]:
def cached_attention(q, k, v, positions, layer_cache):
    """Causal attention of the NEW tokens over the cached tokens and themselves, for the RoPE model.

    q, k, v      (B, n_head, T_new, head_dim)   projections of the new tokens, not rotated yet
    positions    (T_new,)                       positions of the new tokens in the sequence
    layer_cache  None if nothing was processed before, otherwise (K, V): the rotated keys and the values
                 of the T_past earlier tokens, each of shape (B, n_head, T_past, head_dim)
    returns      out (B, n_head, T_new, head_dim), the new cache (K, V) with T_past + T_new tokens,
                 and the attention weights (B, n_head, T_new, T_past + T_new)

    1. rotate q and k with RoPE
    2. append the new k and v to the cached ones (torch.cat on dim=2)
    3. scores of the new queries against ALL the keys, divided by sqrt(head_dim)
    4. mask: new token number t sees every cached token and the new tokens 0 .. t
    5. softmax, then the weighted sum of the values
    """
    # YOUR CODE HERE
    raise NotImplementedError()

The check uses three tokens whose queries and keys are all $(1, 0)$: only RoPE makes them differ. It runs the three
tokens at once, then the first token alone and the two others with its cache, and compares the attention weights.

In [ ]:
q = k = torch.tensor([[[[1.0, 0.0], [1.0, 0.0], [1.0, 0.0]]]])        # 1 sequence, 1 head, 3 tokens, head_dim 2
v = torch.tensor([[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]])
expected = torch.tensor([[1.0000, 0.0000, 0.0000],                     # the attention weights of token 0
                         [0.4195, 0.5805, 0.0000],                     # ... of token 1
                         [0.1758, 0.3457, 0.4785]])                    # ... of token 2

out, cache, weights = cached_attention(q, k, v, torch.tensor([0, 1, 2]), None)                   # all at once
assert torch.allclose(weights[0, 0], expected, atol=1e-4)
assert cache[0].shape == (1, 1, 3, 2) and cache[1].shape == (1, 1, 3, 2)

_, cache_0, _ = cached_attention(q[:, :, :1], k[:, :, :1], v[:, :, :1], torch.tensor([0]), None)  # token 0 first
out_12, cache_012, weights_12 = cached_attention(q[:, :, 1:], k[:, :, 1:], v[:, :, 1:], torch.tensor([1, 2]), cache_0)
assert torch.allclose(weights_12[0, 0], expected[1:], atol=1e-4)
assert torch.allclose(out_12, out[:, :, 1:]) and torch.allclose(cache_012[0], cache[0])

**The model.** Here is the GPT of Lab 10, with a cache. Every layer calls `attention`, which is your `cached_attention`,
and keeps the attention weights of its last call in `weights` (Bonus 1 and 2 look at them).
`model(idx, positions, cache)` returns the logits of the new tokens and the new cache: a list with one `(K, V)` pair
per layer.

In [ ]:
class Block(nn.Module):
    """One layer of the transformer: causal self-attention, then an MLP, each added to its input."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head = n_head
        self.ln_1 = nn.LayerNorm(n_embd, bias=False)
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)     # queries, keys and values of all heads at once
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.ln_2 = nn.LayerNorm(n_embd, bias=False)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
                                 nn.Linear(4 * n_embd, n_embd, bias=False))

    def forward(self, x, positions, layer_cache, attention):
        B, T, C = x.shape
        q, k, v = self.qkv(self.ln_1(x)).split(C, dim=2)          # each (B, T, C)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)   # (B, n_head, T, head_dim)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y, layer_cache, self.weights = attention(q, k, v, positions, layer_cache)
        x = x + self.proj(y.transpose(1, 2).reshape(B, T, C))     # the outputs of the heads side by side
        return x + self.mlp(self.ln_2(x)), layer_cache


class GPT(nn.Module):
    """A character-level GPT with RoPE positions and a KV cache."""

    def __init__(self, vocab_size=65, n_layer=4, n_head=4, n_embd=128):
        super().__init__()
        self.attention = cached_attention                          # the attention step of every layer
        self.wte = nn.Embedding(vocab_size, n_embd)                # one vector per character
        self.blocks = nn.ModuleList([Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd, bias=False)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)   # a score for every possible next character
        self.lm_head.weight = self.wte.weight                      # the same matrix as wte (weight tying, as in GPT-2)

    def forward(self, idx, positions=None, cache=None):
        """idx        (B, T_new) ids of the NEW tokens: the whole sequence when there is no cache
        positions  (T_new,) position of each new token. Default: 0 .. T_new-1, *also when a cache is given*
        cache      None, or the cache returned by the previous call: one (K, V) pair per layer
        returns    the logits (B, T_new, vocab_size) and the cache extended by the new tokens"""
        if positions is None:
            positions = torch.arange(idx.size(1))
        if cache is None:
            cache = [None] * len(self.blocks)
        x = self.wte(idx)                                          # (B, T_new, n_embd)
        new_cache = []
        for block, layer_cache in zip(self.blocks, cache):
            x, layer_cache = block(x, positions, layer_cache, self.attention)
            new_cache.append(layer_cache)
        return self.lm_head(self.ln_f(x)), new_cache

In [ ]:
rope = GPT()
rope.load_state_dict(torch.load("checkpoints/rope.pt"))         # the RoPE model of Lab 10, already trained
print(f"model: {sum(p.numel() for p in rope.parameters()) / 1e6:.2f} M parameters")

In [ ]:
prompt = encode("ROMEO:\nBut soft, what light")[None]             # shape (1, 27): a batch with one sequence
logits, cache = rope(prompt)                                       # no cache given: the model processes everything

print("logits:", tuple(logits.shape), "= (batch, tokens, vocabulary)")
print("cache : a list of", len(cache), "pairs (K, V), one per layer")
print("K of layer 0:", tuple(cache[0][0].shape), "= (batch, heads, tokens, head_dim)")

### 2b. Generating with the cache

`logits, cache = model(idx, positions=None, cache=None)` takes

- `idx`: the ids of the **new** tokens, shape `(1, T_new)`;
- `positions`: the position of each new token, shape `(T_new,)`; if it is not given, the model uses 0, 1, 2, …;
- `cache`: the cache of the tokens processed earlier, or `None`.

It returns the logits of the new tokens, shape `(1, T_new, vocabulary)`, and the cache extended by the new tokens.

In [ ]:
def generate(model, prompt, n_new, use_cache=True):
    """Greedy decoding: append n_new tokens to the prompt, each one the argmax of the logits of the last position.

    prompt (1, T) LongTensor.  Returns a LongTensor of shape (1, T + n_new): the prompt, then the new tokens.
    use_cache=False: every step runs the model on the whole sequence so far.
    use_cache=True:  prefill: one pass over the prompt, which returns the cache;
                     decode:  at every step, feed ONLY the newest token, together with the cache.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
start = encode("ROMEO:\nBut soft")[None]
assert decode(generate(rope, start, 24, use_cache=False)[0]) == "ROMEO:\nBut soft" + "er the stand the content"
assert decode(generate(rope, start, 24, use_cache=True)[0]) == "ROMEO:\nBut soft" + "er the stand the content"

In [ ]:
print(decode(generate(rope, prompt, n_new=L - prompt.size(1))[0]))

### 2c. What does the cache buy?

For timing the weights do not matter, so we use a larger model with random weights (6 layers, 384 dimensions,
10.6 M parameters, the size of nanoGPT's "baby GPT").

In [ ]:
torch.manual_seed(0)
big = GPT(n_layer=6, n_head=6, n_embd=384)
print(f"timing model: {sum(p.numel() for p in big.parameters()) / 1e6:.1f} M parameters, random weights")

In [ ]:
%%time
start = torch.randint(65, (1, 16))

for use_cache in (False, True):
    t0 = time.perf_counter()
    generate(big, start, 200, use_cache=use_cache)[0, -1].item()      # .item() waits for the last token, also on a GPU
    print(f"200 tokens, use_cache={use_cache!s:5}: {time.perf_counter() - t0:6.2f} s")

The plot shows the time to produce **one** more token, as a function of how many tokens are already there:

In [ ]:
def time_per_token(model, lengths, use_cache):
    """Seconds to get the logits of one more token when the context holds `length` tokens, for each length (best of 3).
    Without a cache the model processes the whole context. With a cache, only the last token."""
    seconds = []
    for length in lengths:
        idx = torch.randint(65, (1, length))
        cache = model(idx[:, :-1])[1] if use_cache else None
        new = idx[:, -1:] if use_cache else idx
        runs = []
        for _ in range(3):
            t0 = time.perf_counter()
            model(new, positions=torch.arange(length - new.size(1), length), cache=cache)[0][0, -1, 0].item()
            runs.append(time.perf_counter() - t0)
        seconds.append(min(runs))
    return seconds

In [ ]:
%%time
lengths = [64, 128, 256, 512, 1024, 2048]
plt.plot(lengths, [1e3 * s for s in time_per_token(big, lengths, use_cache=False)], marker="o", label="without KV cache")
plt.plot(lengths, [1e3 * s for s in time_per_token(big, lengths, use_cache=True)], marker="o", label="with KV cache")
plt.xlabel("tokens already in the context")
plt.ylabel("milliseconds per new token")
plt.ylim(bottom=0)
plt.title(f"Time to generate one more token ({device})")
plt.legend()
plt.show()

Prefill and decode do not cost the same per token:

In [ ]:
T = 512
tokens = torch.randint(65, (1, T + 20))

prefill = float("inf")
for _ in range(3):                                                     # best of 3 runs
    t0 = time.perf_counter()
    logits, cache = big(tokens[:, :T])                                 # prefill: 512 tokens in one pass
    logits[0, -1, 0].item()                                            # wait until the result is there
    prefill = min(prefill, time.perf_counter() - t0)

t0 = time.perf_counter()
for t in range(T, T + 20):                                             # decode: 20 tokens, one pass each
    logits, cache = big(tokens[:, t : t + 1], positions=torch.tensor([t]), cache=cache)
logits[0, -1, 0].item()
decode_time = (time.perf_counter() - t0) / 20

print(f"prefill: {T} tokens in {1e3 * prefill:6.1f} ms  = {1e3 * prefill / T:5.3f} ms per token")
print(f"decode : 1 token in   {1e3 * decode_time:6.2f} ms")
print(f"one decoded token costs as much as {decode_time / (prefill / T):.0f} prefilled tokens")

**Question 2.**
1. Which phase is compute-bound and which is memory-bound, prefill or decode? Why? Use it to explain the plot: from
   which length on does the cache pay off on your machine?
2. What exactly goes wrong if you write `model(next_token, cache=cache)` and forget `positions`?

*Your answer here*

## Exercise 3 — How big is the cache?

> Lecture 11, *KV memory*, and *Smaller caches*: MQA, GQA, MLA.

For every token the cache holds one key and one value per layer and per **key/value head**:

$$\text{bytes per token}=2\;(K\text{ and }V)\times n_\text{layers}\times n_\text{kv heads}\times d_\text{head}\times\text{bytes per element}.$$

With grouped-query attention (GQA) several query heads share one key/value head; with multi-query attention (MQA) all of
them share a single one.

### 3a. The cache of our model

In [ ]:
cache = rope(val_data[:1000][None])[1]                             # the cache of 1000 tokens
bytes_per_token = sum(K.nbytes + V.nbytes for K, V in cache) / 1000   # .nbytes: the memory taken by a tensor
print("K of one layer:", tuple(cache[0][0].shape), cache[0][0].dtype)
print(f"the cache takes {bytes_per_token:.0f} bytes per token")

### 3b. How many tokens fit?

A serving engine such as vLLM does not grow the cache with `torch.cat`. It reserves the GPU memory of the cache in
blocks of 16 tokens and hands them out to the requests. How long the context can be is then simply how many blocks fit
before the GPU runs out of memory.

**This part needs a GPU.** To keep the experiment fast, and harmless on a GPU that other people use too,
`tokens_in_1_gib` below gives PyTorch only 1 GiB more than what the notebook already holds
(`torch.cuda.set_per_process_memory_fraction`). Beyond that, PyTorch raises `torch.cuda.OutOfMemoryError` exactly as if
the GPU were full.

**Your task.** Allocate cache blocks until the GPU is full.

In [ ]:
def tokens_before_oom(n_layer, n_kv_head, head_dim, dtype, block_size=16):
    """Allocate KV-cache blocks until the GPU runs out of memory, and return the number of tokens that fit.

    One block holds the keys and the values of block_size tokens in every layer: a tensor of shape
    (n_layer, 2, n_kv_head, block_size, head_dim), created with torch.empty(..., dtype=dtype).
    Keep the blocks in a list (or they are freed at once) until torch.cuda.OutOfMemoryError is raised.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
def tokens_in_1_gib(configs):
    """Run tokens_before_oom for each config with 1 GiB of free GPU memory, print the results and return them."""
    torch.cuda.empty_cache()                                         # give back the memory that no tensor uses
    limit = torch.cuda.memory_reserved() + 2**30                     # what the notebook holds already, plus 1 GiB
    torch.cuda.set_per_process_memory_fraction(limit / torch.cuda.get_device_properties(device).total_memory)
    fits = {name: tokens_before_oom(**config) for name, config in configs.items()}
    torch.cuda.set_per_process_memory_fraction(1.0)                  # remove the limit
    for name, n in fits.items():
        print(f"{name:40s} {n:10,d} tokens in 1 GiB")
    return fits

Our model with 4, 2 and 1 key/value heads (the 4 query heads stay):

In [ ]:
fits = tokens_in_1_gib({
    "MHA: 4 query heads, 4 KV heads": dict(n_layer=4, n_kv_head=4, head_dim=32, dtype=torch.float32),
    "GQA: 4 query heads, 2 KV heads": dict(n_layer=4, n_kv_head=2, head_dim=32, dtype=torch.float32),
    "MQA: 4 query heads, 1 KV head":  dict(n_layer=4, n_kv_head=1, head_dim=32, dtype=torch.float32),
})
mha, gqa, mqa = fits.values()
assert 0.99 < mha * bytes_per_token / 2**30 < 1.01          # 1 GiB divided by the bytes per token measured in 3a
assert 0.99 < gqa / (2 * mha) < 1.01 and 0.99 < mqa / (4 * mha) < 1.01

**Gemma 2 27B.** The lecture slide computes `46 × 32 × 128 × 2 × 2` bytes per token: 46 layers, 32 heads of dimension
128, K and V, 16-bit floats. The table on the same slide lists **16 KV heads**.

In [ ]:
fits = tokens_in_1_gib({
    "(a) as on the slide: 32 heads":         dict(n_layer=46, n_kv_head=32, head_dim=128, dtype=torch.bfloat16),
    "(b) with the 16 KV heads of the table": dict(n_layer=46, n_kv_head=16, head_dim=128, dtype=torch.bfloat16),
})
for name, n in fits.items():
    print(f"{name:40s} (c) about {80 * n:,} tokens in 80 GiB")

**MLA (DeepSeek).** Multi-head latent attention goes further: it caches one small latent vector $c$ per token and layer,
and keys and values are linear maps of it, $k=W_k\,c$ and $v=W_v\,c$. Nothing needs to be rebuilt at inference time,
because the key matrix can be moved to the query:

$$q^\top(W_k\,c)=(W_k^\top q)^\top c .$$

In [ ]:
# DeepSeek-V2, numbers from its paper: 60 layers, 128 heads of dimension 128, 16-bit floats.
# MLA caches a latent of 512 numbers per layer, plus 64 numbers for a small key that carries RoPE: 576 numbers,
# as many as the keys and values of a single head of dimension 288 (2 x 288 = 576).
fits = tokens_in_1_gib({
    "caching K and V of 128 heads": dict(n_layer=60, n_kv_head=128, head_dim=128, dtype=torch.bfloat16),
    "caching the latent c":         dict(n_layer=60, n_kv_head=1, head_dim=288, dtype=torch.bfloat16),
})
print(f"MLA fits {fits['caching the latent c'] / fits['caching K and V of 128 heads']:.0f} times more tokens")

torch.manual_seed(0)
d_c, d_kv = 16, 64                              # size of the latent, size of all the heads together
W_k = torch.randn(d_kv, d_c)
c, q = torch.randn(d_c), torch.randn(d_kv)
print(f"\nq . (W_k c) = {(q @ (W_k @ c)).item():.4f}       (W_k^T q) . c = {((W_k.T @ q) @ c).item():.4f}")

**Question 3.** The slide multiplies by 32 heads, the table says 16 KV heads. Which number decides how many tokens fit,
and what changes? Is "114k tokens on an 80 GB A100" realistic?

*Your answer here*

## Exercise 4 — Prefix caching across conversations

> Lecture 11, *Cross-conversation (prompt) caching*.

Thousands of requests start with the same system prompt. Causality helps once more: the keys and values of the first
$n$ tokens depend on these $n$ tokens only. So **the cache of a prompt is a valid cache for every prefix of that
prompt**, and any request that starts with the same $n$ tokens can reuse it.

A `PrefixCache` remembers pairs (tokens, cache). For a new prompt it finds the stored sequence that shares the longest
prefix with it and returns the cache of that prefix. Only the rest of the prompt has to be prefilled.

**Copy, do not alias.** Our model never writes into a cache tensor: it builds new tensors with `torch.cat`. Real
inference engines pre-allocate the cache and write into it in place (3b), and a conversation keeps appending to the
cache it was given. A prefix cache must therefore own its data: copy on the way in and copy on the way out
(`.clone()`). The check below overwrites the tensors it gets, as such an engine would.

### 4a. The prefix cache

In [ ]:
def common_prefix_length(a, b):
    """Number of leading tokens that the 1-D tensors a and b have in common."""
    n = min(len(a), len(b))
    different = (a[:n] != b[:n]).nonzero()
    return n if len(different) == 0 else int(different[0])

common_prefix_length(encode("To be, or not to be"), encode("To be or not"))

In [ ]:
class PrefixCache:
    """Remembers the KV cache of the prompts it has seen, so that later prompts can reuse a common prefix."""

    def __init__(self):
        self.entries = []                        # list of (tokens, cache) pairs

    def insert(self, tokens, cache):
        """Remember that `cache` is the KV cache of the 1-D tensor `tokens`. Store COPIES of both."""
        # YOUR CODE HERE
        raise NotImplementedError()

    def lookup(self, tokens):
        """Find the stored sequence that shares the longest prefix with `tokens`.

        Returns (n, cache): the length n of the common prefix and a COPY of the cache of these n tokens,
        that is every K and V cut down to its first n tokens (K[:, :, :n]). Returns (0, None) if nothing matches.
        """
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
def prefill_with_prefix_cache(model, prefix_cache, tokens):
    """Prefill the 1-D tensor `tokens`, reusing the longest cached prefix.

    Returns (logits of the last token, cache of the whole prompt, n), where n is the number of tokens taken
    from the prefix cache. The model must process at least the last token to produce logits, so n is at
    most len(tokens) - 1. At the end, insert the whole prompt and its cache into the prefix cache.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
system = encode("First Citizen:\nBefore we proceed any further, hear me speak.\n\n")
a = torch.cat([system, encode("All:\nSpeak, speak.")])
b = torch.cat([system, encode("All:\nResolved. resolved.")])     # a and b share the system prompt and "All:\n"

prefix_cache = PrefixCache()
assert prefix_cache.lookup(a) == (0, None)
logits, _, n = prefill_with_prefix_cache(rope, prefix_cache, a)
assert n == 0 and torch.allclose(logits, rope(a[None])[0][0, -1], atol=1e-4)
logits, _, n = prefill_with_prefix_cache(rope, prefix_cache, b)
assert n == len(system) + 5 and torch.allclose(logits, rope(b[None])[0][0, -1], atol=1e-4)
logits, _, n = prefill_with_prefix_cache(rope, prefix_cache, b)  # the same prompt again
assert n == len(b) - 1 and torch.allclose(logits, rope(b[None])[0][0, -1], atol=1e-4)

prefix_cache.lookup(b)[1][0][0].fill_(float("nan"))              # overwrite what lookup returned ...
assert not prefix_cache.lookup(b)[1][0][0].isnan().any()         # ... the stored cache is not touched
cache_a = rope(a[None])[1]
prefix_cache = PrefixCache()
prefix_cache.insert(a, cache_a)
cache_a[0][0].fill_(float("nan"))                                # overwrite what was inserted ...
assert not prefix_cache.lookup(a)[1][0][0].isnan().any()         # ... the stored cache is not touched

### 4b. Three ways to write the same prompts

A small "application": a fixed system prompt of about 460 characters and 20 requests that differ in two words.
The same 20 requests are written in three ways.

- **A, variable part first:** the request, then the system prompt.
- **B, stable part first:** the system prompt, then the request.
- **C, stable part first and templated:** the system prompt, the request with placeholders, and the two words at the very
  end. This is the lecture's *"Please book flight from X to Y; x=ZH, y=SFO"*.

(The prompts are about ten times longer than the context the model was trained on, so what it predicts here is
worthless. Lab 10 explains why. We only measure time.)

In [ ]:
system = text[: text.index("\n\n", 450) + 2]            # the "system prompt": the first lines of the play
names = ["Romeo", "Juliet", "Hamlet", "Portia", "Brutus", "Viola", "Oberon", "Titania", "Prospero", "Miranda",
         "Orlando", "Rosalind", "Antonio", "Bassanio", "Shylock", "Cordelia", "Edgar", "Kent", "Banquo", "Duncan"]
places = ["Verona", "Mantua", "Padua", "Venice", "Athens", "Rome", "Milan", "Naples", "Messina", "Vienna",
          "Windsor", "Dover", "York", "Calais", "Paris", "Ephesus", "Tyre", "Cyprus", "Troy", "Egypt"]

request = "MESSENGER:\n{x} of {y} craves audience, and bids me ask what cause hath moved the people to this mutiny.\n"

def design_A(x, y):
    return request.format(x=x, y=y) + "\n" + system

def design_B(x, y):
    return system + request.format(x=x, y=y) + "\n"

def design_C(x, y):
    return system + request.format(x="X", y="Y") + f"X is {x}; Y is {y}.\n\n"

print(f"system prompt: {len(system)} characters. One request in design C:\n")
print(design_C(names[0], places[0])[len(system):])

In [ ]:
def measure(design, repeats=3):
    """Send the 20 requests through a fresh PrefixCache. Returns the share of prompt tokens that came from
    the cache and the median time to first token of requests 2 to 20 (best of `repeats` runs)."""
    prompts = [encode(design(x, y)) for x, y in zip(names, places)]
    runs = []
    for _ in range(repeats):
        prefix_cache, cached, ttft = PrefixCache(), 0, []
        for tokens in prompts:
            t0 = time.perf_counter()
            logits, cache, n = prefill_with_prefix_cache(rope, prefix_cache, tokens)   # these logits give the first token
            logits[0].item()                                                          # wait until they are there
            ttft.append(time.perf_counter() - t0)
            cached += n
        runs.append((statistics.median(ttft[1:]), cached / sum(len(p) for p in prompts)))
    ttft, share = min(runs)
    return share, ttft

stats = {"A: variable first": measure(design_A), "B: stable first": measure(design_B), "C: templated": measure(design_C)}

full = []                                      # for comparison: a plain prefill of the design-B prompts, no prefix cache
for x, y in zip(names, places):
    tokens = encode(design_B(x, y))
    t0 = time.perf_counter()
    rope(tokens[None])[0][0, -1, 0].item()
    full.append(time.perf_counter() - t0)

for name, (share, ttft) in stats.items():
    print(f"{name:20s} {100 * share:5.1f} % of the prompt tokens from the cache    time to first token {1e3 * ttft:6.2f} ms")
print(f"{'no prefix cache':20s} {0:5.1f} %{'':41s}time to first token {1e3 * statistics.median(full):6.2f} ms")

plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.bar(list(stats), [100 * share for share, _ in stats.values()])
plt.title("% of the prompt tokens from the cache")
plt.subplot(1, 2, 2)
plt.bar(list(stats), [1e3 * ttft for _, ttft in stats.values()])
plt.title(f"time to first token, ms ({device})")
plt.show()

**Question 4.**
1. Designs B and C both put the stable system prompt first. Why is C still better?
2. API providers bill cached input tokens at a fraction of the normal price. With your measurements in mind, how should
   the system prompt of an agent be organised? Where do the current date, the user's name and the results of tool calls go?

*Your answer here*

## Bonus 1 — Streaming past the context: windows and attention sinks

> Lecture 11, *Bounded caches*: sliding-window attention and StreamingLLM.

A cache that only grows is no good for a stream. The StreamingLLM figure of the lecture compares three policies on a
long text: **dense** attention (keep everything), **window** attention (keep the most recent tokens) and
**StreamingLLM** (keep the first few tokens, the "attention sinks", and the most recent ones). In the paper, dense
attention fails once the text is longer than the training length, window attention collapses as soon as the first
tokens are evicted, and keeping four sink tokens repairs it.

We tried this on our tiny model before writing the lab, and one of the paper's three results does not reproduce. This is
why the exercise is a bonus. It is still worth doing: finding out which result is missing, and why, shows what an
attention sink is.

**Your task.** Write the two eviction policies. A cache is a list with one `(K, V)` pair per layer. The tokens are along
dimension 2.

In [ ]:
def evict_window(cache, budget):
    """Keep only the `budget` most recent tokens, in every layer."""
    # YOUR CODE HERE
    raise NotImplementedError()


def evict_sinks_window(cache, budget, n_sink=4):
    """Keep the first n_sink tokens (the "sinks") and the budget - n_sink most recent tokens.
    A cache that holds `budget` tokens or fewer is returned unchanged."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
numbers = torch.arange(10.0).view(1, 1, 10, 1)            # a K (or V) of 10 tokens in which token t holds the number t
cache = [(numbers, numbers), (numbers, numbers)]           # 2 layers
assert [K.flatten().tolist() for K, V in evict_window(cache, 4)] == [[6, 7, 8, 9]] * 2
assert [V.flatten().tolist() for K, V in evict_sinks_window(cache, 6, n_sink=2)] == [[0, 1, 6, 7, 8, 9]] * 2
assert evict_sinks_window(cache, 12)[0][0].flatten().tolist() == list(range(10))    # within the budget: unchanged

`streaming_loss(model, tokens, evict)` feeds a text one character at a time with a KV cache, calls `evict(cache)`
before every new character, and returns the loss at every position.

With sinks, the cache has a hole between the sinks and the window. StreamingLLM numbers the tokens by their place *in
the cache*, not in the text. `attention_by_slot` does that: it caches the keys before the rotation and rotates them at
each step according to their slot in the cache. A model uses it instead of your `cached_attention` once we set its
`attention` to it.

In [ ]:
def streaming_loss(model, tokens, evict):
    """Read `tokens` one at a time with a KV cache, calling evict(cache) before every new token. Token t is always given
    the position t. Returns the loss of the prediction of every next token, shape (len(tokens) - 1,)."""
    cache, losses = None, []
    for t in range(len(tokens) - 1):
        if t > 0:
            cache = evict(cache)
        logits, cache = model(tokens[None, t : t + 1], positions=torch.tensor([t]), cache=cache)
        losses.append(F.cross_entropy(logits[0, -1], tokens[t + 1]))
    return torch.stack(losses)


def attention_by_slot(q, k, v, positions, layer_cache):
    """The attention step of StreamingLLM: the cache holds the keys BEFORE the rotation, and every key is rotated by
    its slot in the cache, at every step. The new tokens take the last slots: `positions` is not used."""
    if layer_cache is not None:
        k = torch.cat([layer_cache[0], k], dim=2)
        v = torch.cat([layer_cache[1], v], dim=2)
    T_new, T_all, head_dim = q.size(2), k.size(2), q.size(-1)
    angles = torch.arange(T_all)[:, None] * rope_theta(head_dim)[None, :]       # slot j: the angles j * theta_i
    scores = apply_rope(q, angles[-T_new:]) @ apply_rope(k, angles).transpose(-2, -1) / math.sqrt(head_dim)
    future = torch.ones(T_new, T_all, dtype=torch.bool).triu(diagonal=T_all - T_new + 1)
    weights = F.softmax(scores.masked_fill(future, float("-inf")), dim=-1)
    return weights @ v, (k, v), weights

In [ ]:
%%time
stream = val_data[:4000]                 # 4000 characters that the model has never seen
budget = L - 1                           # 63 cached tokens + the current one = 64, the training context
rope_slots = copy.deepcopy(rope)
rope_slots.attention = attention_by_slot

curves = {
    "dense (no eviction)": streaming_loss(rope, stream, lambda cache: cache),
    "window":              streaming_loss(rope, stream, lambda cache: evict_window(cache, budget)),
    "4 sinks + window":    streaming_loss(rope_slots, stream, lambda cache: evict_sinks_window(cache, budget)),
}
for name, losses in curves.items():
    plt.plot(range(199, len(losses)), losses.unfold(0, 200, 1).mean(dim=1).tolist(), label=name)   # mean of 200 characters
plt.axvline(L, color="gray", linestyle="--", linewidth=1)                       # the training length
plt.xlabel("position in the stream (characters)")
plt.ylabel("loss, mean over the last 200 characters")
plt.title("Streaming a long text through the model")
plt.legend()
plt.show()
for name, losses in curves.items():
    print(f"{name:22s} mean loss after the first {L} characters: {losses[L:].mean():.3f}")

Why do the sinks change nothing? Look at where the model puts its attention:

In [ ]:
windows = torch.stack([val_data[s : s + L] for s in range(0, 256 * L, L)])      # 256 validation windows of 64 characters
rope(windows)                                        # every layer keeps the attention weights of this call

print("share of the attention of the tokens at positions 16 .. 63 that goes to")
print("layer     the first token     the 8 most recent tokens")
for layer, block in enumerate(rope.blocks):
    w = block.weights                                # (windows, heads, queries, keys)
    first = w[:, :, 16:, 0].mean().item()
    recent = sum(w[:, :, t, t - 7 : t + 1].sum(-1).mean().item() for t in range(16, L)) / (L - 16)
    print(f"  {layer}           {100 * first:5.1f} %               {100 * recent:5.1f} %")

And what if the sinks keep their original positions (no renumbering inside the cache)?

In [ ]:
original = streaming_loss(rope, stream, lambda cache: evict_sinks_window(cache, budget))
print(f"4 sinks + window, original positions: mean loss after the first {L} characters: {original[L:].mean():.3f}")

The paper also proposes to train the model with a dedicated sink token. We did: `checkpoints/rope_bos.pt` is the same
model, trained with a BOS token in front of every training window. We put the BOS in front of the stream as well.
Is its BOS a sink?

In [ ]:
%%time
bos_model = GPT(vocab_size=66)                                     # one more token than characters: the BOS, id 65
bos_model.load_state_dict(torch.load("checkpoints/rope_bos.pt"))
bos_slots = copy.deepcopy(bos_model)
bos_slots.attention = attention_by_slot

bos_stream = torch.cat([torch.tensor([65]), stream])              # the BOS, then the text
window = streaming_loss(bos_model, bos_stream, lambda cache: evict_window(cache, budget))
sink = streaming_loss(bos_slots, bos_stream, lambda cache: evict_sinks_window(cache, budget, n_sink=1))
print(f"model trained with BOS: window {window[L:].mean():.3f}     BOS kept as a sink + window {sink[L:].mean():.3f}")

**Question B1.**
1. Dense attention has *more* information than the window. Why is it so much worse?
2. In the paper, window attention collapses and the sinks repair it. Here the two curves coincide. Use the attention
   table to explain the difference.
3. Why does StreamingLLM renumber the positions inside the cache?

*Your answer here*

## Bonus 2 — H2O: keep the heavy hitters

> Lecture 11, *KV pruning*: Scissorhands and H2O.

H2O observes that in large models a few tokens receive most of the attention. It keeps, in each layer, the tokens with
the highest **accumulated attention** (the "heavy hitters") plus a window of recent tokens, and evicts the rest.

**Your task.** Decide which cache slots survive. (This bonus uses `evict_window`, `rope_slots` and `budget` from Bonus 1.)

In [ ]:
def h2o_keep(scores, budget, n_recent):
    """Which tokens stay in the cache of one layer?

    scores    (T,) attention that each cached token has received so far, summed over heads and over time
    returns   a sorted 1-D LongTensor with the indices to keep: the n_recent most recent tokens, and among
              the older ones the budget - n_recent tokens with the highest scores.
              If T <= budget, everything is kept.
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
scores = torch.tensor([0.1, 5.0, 0.2, 3.0, 0.3, 0.9, 4.0, 0.6, 0.0, 0.05])
assert h2o_keep(scores[:4], 6, 2).tolist() == [0, 1, 2, 3]          # fewer tokens than the budget: keep everything
assert h2o_keep(scores, 6, 2).tolist() == [1, 3, 5, 6, 8, 9]        # 4 heavy hitters among tokens 0..7, then the last 2
assert h2o_keep(scores, 4, 3).tolist() == [1, 7, 8, 9]              # 1 heavy hitter among tokens 0..6, then the last 3

In [ ]:
def stream_h2o(model, tokens, budget, n_recent):
    """Like streaming_loss, with the H2O policy: each layer keeps its own heavy hitters."""
    cache, losses = None, []
    scores = [torch.zeros(0) for _ in model.blocks]                     # one score per cached token, per layer
    for t in range(len(tokens) - 1):
        logits, cache = model(tokens[None, t : t + 1], positions=torch.tensor([t]), cache=cache)
        losses.append(F.cross_entropy(logits[0, -1], tokens[t + 1]))
        for layer, block in enumerate(model.blocks):
            attention = block.weights[0, :, -1].sum(dim=0)               # what the new token paid to each cached token
            scores[layer] = torch.cat([scores[layer], torch.zeros(1)]) + attention
            keep = h2o_keep(scores[layer], budget, n_recent)
            K, V = cache[layer]
            cache[layer] = (K[:, :, keep], V[:, :, keep])
            scores[layer] = scores[layer][keep]
    return torch.stack(losses)

In [ ]:
%%time
short = val_data[:2000]
print("mean loss after the first 64 characters, at the same cache budget")
for budget in (63, 16):
    window = streaming_loss(rope, short, lambda cache: evict_window(cache, budget))[L:].mean()
    h2o = stream_h2o(rope, short, budget, n_recent=budget // 2)[L:].mean()
    h2o_slots = stream_h2o(rope_slots, short, budget, n_recent=budget // 2)[L:].mean()
    print(f"  budget {budget:2d}:   window {window:.3f}    H2O ({budget - budget // 2} heavy + {budget // 2} recent) {h2o:.3f}"
          f"    H2O, positions renumbered {h2o_slots:.3f}")

**Question B2.** At the same budget, is H2O better than the plain window on this model? Explain the result with what
you measured in Bonus 1.

*Your answer here*

## Bonus 3 — MLA: attention without rebuilding keys and values

> Lecture 11, *Smaller caches*: MLA.

Exercise 3 checked $q^\top(W_k c)=(W_k^\top q)^\top c$ for the scores. The value side works in the same way: the output
is $\sum_j w_j\,(W_v c_j)=W_v\big(\sum_j w_j\,c_j\big)$, so $W_v$ is applied once, after the weighted sum of the latents.

**Your task.** Compute attention from the cached latents `C` without ever building the keys `C @ W_k.T` or the values
`C @ W_v.T`.

In [ ]:
def mla_attention(q, C, W_k, W_v):
    """Attention of one query over cached latents.

    q (d,)    C (L, d_c): one latent per cached token    W_k, W_v (d, d_c): k_j = W_k c_j and v_j = W_v c_j
    returns (d,):  sum_j softmax_j(q . k_j / sqrt(d)) v_j, computed without any tensor of shape (L, d)
    """
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
torch.manual_seed(0)
q, C = torch.randn(32), torch.randn(50, 8)                 # one query, 50 cached latents of size 8
W_k, W_v = torch.randn(32, 8), torch.randn(32, 8)
assert torch.allclose(mla_attention(q, C, W_k, W_v), attention_naive(q, C @ W_k.T, C @ W_v.T), atol=1e-4)  # keys and values rebuilt

**Question B3.** DeepSeek-V2 caches 512 numbers for the latent and 64 more for a separate small key that carries RoPE.
Why can RoPE not simply be applied to the keys $W_k c$? (Think of what you just did with $W_k$, and of Lab 10.)

*Your answer here*

---
**That's it.** Every trick of this lab rests on one property: with a causal mask, the keys and values of a token depend
only on the tokens before it. That is why they can be cached, shared between conversations, and thrown away one by one.